In [11]:
import pandas as pd
from data_profiling import ProfileReport
import numpy as np

# Partie 1 Profiling & audit

In [2]:
df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")
df.describe()

C:\Users\paulk\AppData\Local\Temp\ipykernel_53052\1620990980.py:1: DtypeWarning: Columns (12,18,19,20,21,22,24,25,29,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv")


,siren_amenageur,nbre_pdc,puissance_nominale,consolidated_longitude,consolidated_latitude,consolidated_code_postal
count,1.362470e+05,227232.000000,227232.000000,227232.000000,227232.000000,134362.000000
mean,6.924200e+08,14.983180,102.212347,2.678738,46.733937,52603.766683
std,2.572975e+08,47.103571,579.594838,4.529209,4.034939,26780.472001
min,0.000000e+00,1.000000,0.000000,-149.905377,-44.996198,1000.000000
25%,5.243353e+08,2.000000,22.000000,0.774255,44.871519,31112.500000
50%,8.427185e+08,4.000000,22.000000,2.412432,47.340547,57160.000000
75%,8.951636e+08,9.000000,100.000000,4.836760,48.865021,76410.000000
max,9.921637e+08,505.000000,160000.000000,166.462000,61.520355,97418.000000


In [3]:
df.columns

Index(['nom_amenageur', 'siren_amenageur', 'contact_amenageur',
       'nom_operateur', 'contact_operateur', 'telephone_operateur',
       'nom_enseigne', 'id_station_itinerance', 'id_station_local',
       'nom_station', 'implantation_station', 'adresse_station',
       'code_insee_commune', 'coordonneesXY', 'nbre_pdc', 'id_pdc_itinerance',
       'id_pdc_local', 'puissance_nominale', 'prise_type_ef', 'prise_type_2',
       'prise_type_combo_ccs', 'prise_type_chademo', 'prise_type_autre',
       'gratuit', 'paiement_acte', 'paiement_cb', 'paiement_autre',
       'tarification', 'condition_acces', 'reservation', 'horaires',
       'accessibilite_pmr', 'restriction_gabarit', 'station_deux_roues',
       'raccordement', 'num_pdl', 'date_mise_en_service', 'observations',
       'date_maj', 'cable_t2_attache', 'last_modified', 'datagouv_dataset_id',
       'datagouv_resource_id', 'datagouv_organization_or_owner', 'created_at',
       'consolidated_longitude', 'consolidated_latitude',
     

In [4]:
profile = ProfileReport(df, title="Profiling Report")

In [5]:
report_path = "rapport_data_profiling.html"
profile.to_file(report_path)

print(f"Rapport généré : {report_path}")

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 136 (\x88) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 143 (\x8f) missing from font(s) Arial.
  plt.savefig(
c:\Users\paulk\qual_donnees\coda_qualite_donnees\venv\Lib\site-packages\data_profiling\visualisation\utils.py:73: UserWarning: Glyph 142 (\x8e) missing from font(s) Arial.
  plt.savefig(
E

Rapport généré : rapport_data_profiling.html


In [6]:
def quality_score(df):
    total_cells = df.shape[0] * df.shape[1]

    missing = df.isna().sum().sum()
    completeness = 1 - missing / total_cells

    duplicates = df.duplicated().sum() / len(df)

    numeric_outliers = 0
    for col in df.select_dtypes("number"):
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        outliers = ((df[col] < q1 - 1.5 * iqr) | (df[col] > q3 + 1.5 * iqr)).sum()
        numeric_outliers += outliers

    outlier_ratio = numeric_outliers / total_cells

    score = (
        0.5 * completeness +
        0.3 * (1 - duplicates) +
        0.2 * (1 - outlier_ratio)
    )

    return round(score * 100, 2)

print(quality_score(df))

93.77


93.77% des données semblent de qualité en se basant sur le nombre de doublons, de valeurs aberrantes et de valeurs manquantes.

In [28]:
df["siren_amenageur"] = (
    df["siren_amenageur"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

Le type de "siren_amenageur" était un float, on l'a converti en str pour pouvoir effectuer du Regex dessus et checker sa validité 

In [29]:
audit_rows = []

# -------------------------
# 1. COMPLETUDE
# -------------------------
for col in df.columns:
    missing_rate = df[col].isna().mean()

    audit_rows.append({
        "dimension": "completude",
        "colonne": col,
        "pct_problemes": round(missing_rate * 100, 2),
        "gravite": "haute" if missing_rate > 0.3 else "moyenne" if missing_rate > 0.1 else "faible"
    })


# -------------------------
# 2. UNICITE (ID station)
# -------------------------
dup_rate = df.duplicated(subset=["id_station_itinerance"]).mean()

audit_rows.append({
    "dimension": "unicite",
    "colonne": "id_station_itinerance",
    "pct_problemes": round(dup_rate * 100, 2),
    "gravite": "haute" if dup_rate > 0.05 else "moyenne" if dup_rate > 0.01 else "faible"
})


# -------------------------
# 3. VALIDITE - SIREN
# -------------------------
siren_invalid = ~df["siren_amenageur"].astype(str).str.match(r"^\d{9}$")
siren_rate = siren_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "siren_amenageur",
    "pct_problemes": round(siren_rate * 100, 2),
    "gravite": "haute" if siren_rate > 0.1 else "moyenne" if siren_rate > 0.02 else "faible"
})


# -------------------------
# 4. COHERENCE GEO
# -------------------------
geo_invalid = ~(
    df["consolidated_latitude"].between(41, 51) &
    df["consolidated_longitude"].between(-5, 10)
)

geo_rate = geo_invalid.mean()

audit_rows.append({
    "dimension": "coherence_geo",
    "colonne": "latitude_longitude",
    "pct_problemes": round(geo_rate * 100, 2),
    "gravite": "haute" if geo_rate > 0.2 else "moyenne" if geo_rate > 0.05 else "faible"
})


# -------------------------
# 5. VALIDITE PUISSANCE
# -------------------------
power_invalid = df["puissance_nominale"] <= 0
power_rate = power_invalid.mean()

audit_rows.append({
    "dimension": "validite",
    "colonne": "puissance_nominale",
    "pct_problemes": round(power_rate * 100, 2),
    "gravite": "haute" if power_rate > 0.1 else "moyenne" if power_rate > 0.02 else "faible"
})


# -------------------------
# RESULTAT FINAL
# -------------------------
audit_df = pd.DataFrame(audit_rows)
audit_df.sort_values("pct_problemes", ascending=False)

,dimension,colonne,pct_problemes,gravite
37,completude,observations,79.53,haute
27,completude,tarification,75.20,haute
52,unicite,id_station_itinerance,71.86,haute
35,completude,num_pdl,48.96,haute
39,completude,cable_t2_attache,46.97,haute
47,completude,consolidated_code_postal,40.87,haute
36,completude,date_mise_en_service,40.61,haute
53,validite,siren_amenageur,40.05,haute
34,completude,raccordement,34.90,haute
16,completude,id_pdc_local,34.62,haute


In [30]:
audit_df.groupby("dimension")["pct_problemes"].mean()

dimension
coherence_geo     0.530000
completude       11.496538
unicite          71.860000
validite         21.185000
Name: pct_problemes, dtype: float64

On voit que la catégorie qui pose le plus de problèmes est l'unicité avec 72% de données non uniques par colonnes en moyenne puis quelques problèmes de complétude, validité et de cohérence_geo